In [0]:
# COMMAND ----------
dbutils.widgets.dropdown("load_type", "incremental", ["full_backfill", "incremental"])
dbutils.widgets.text("target_dates", "24.09.2026,1.10.2026") 
dbutils.widgets.text("wfp_csv_path", "/Volumes/workspace/default/spi_data/samples/wfpvam_foodprices.csv")
dbutils.widgets.text("base_dir", "/Volumes/workspace/default/spi_data/samples")

load_type = dbutils.widgets.get("load_type")
target_dates_str = dbutils.widgets.get("target_dates")
wfp_csv_path = dbutils.widgets.get("wfp_csv_path")
base_dir = dbutils.widgets.get("base_dir")

target_dates = [d.strip() for d in target_dates_str.split(",")]
batch_id = f"batch_{target_dates[-1]}_{load_type}" # Use the latest date for the batch ID

# COMMAND ----------
print("--- Step 1: Running Setup ---")
import time
for attempt in range(3):
    try:
        dbutils.notebook.run("Schemas", 600)
        break
    except Exception as e:
        if "REQUEST_LIMIT_EXCEEDED" in str(e) and attempt < 2:
            print(f"Previous run still active, waiting 60s (attempt {attempt + 1}/3)...")
            time.sleep(60)
        else:
            raise

# COMMAND ----------
print("--- Step 2: Processing WFP Bulk Data (Running Once) ---")
print("Extracting Bronze WFP...")
dbutils.notebook.run("bronzewfp", 1200, {
    "wfp_file_path": wfp_csv_path
})

print("Transforming Silver WFP...")
dbutils.notebook.run("silverwfp", 1200, {
    "batch_id": batch_id
})

# COMMAND ----------
print("--- Step 3: Processing PBS Weekly Data ---")
for target_date in target_dates:
    print(f"Processing PBS for week: {target_date}")
    
    dbutils.notebook.run("bronzepbs", 600, {
        "base_dir": base_dir, 
        "week_ending_date": target_date
    })
    
    dbutils.notebook.run("silverpbs", 600, {
        "week_ending_date": target_date
    })

print("\n--- Pipeline Execution Complete ---")

# COMMAND ----------
# MAGIC %sql
# MAGIC SELECT * FROM workspace.default.pipeline_execution_logs ORDER BY start_time DESC;

--- Step 1: Running Setup ---
--- Step 2: Processing WFP Bulk Data (Running Once) ---
Extracting Bronze WFP...
Transforming Silver WFP...
--- Step 3: Processing PBS Weekly Data ---
Processing PBS for week: 24.09.2026

--- Pipeline Execution Complete ---


In [0]:
# CELL 3: Final Audit Verification
# This lets you visually confirm that every step ran and see how many rows were inserted.
display(spark.sql("SELECT layer, file_processed, start_time, status, rows_processed FROM default.pipeline_execution_logs ORDER BY start_time DESC"))

layer,file_processed,start_time,status,rows_processed
Bronze-to-Silver-PBS,Week_24.09.2026,2026-10-07T19:47:35.568Z,Success,191
Raw-to-Bronze-PBS,Week_24.09.2026,2026-10-07T19:47:08.910Z,Success,218
Bronze-to-Silver-WFP,batch_24.09.2026_incremental,2026-10-07T19:46:33.000Z,Success,46710
Raw-to-Bronze-WFP,wfp_sample.csv,2026-10-07T19:45:52.066Z,Success,2050638
Bronze-to-Silver-WFP,manual_run,2026-10-07T19:28:59.326Z,Success,37368
Bronze-to-Silver-PBS,Week_24.09.2026,2026-10-07T19:28:31.126Z,Success,191
Raw-to-Bronze-WFP,wfp_sample.csv,2026-10-07T19:28:05.268Z,Success,2050638
Raw-to-Bronze-PBS,Week_24.09.2026,2026-10-07T19:27:48.184Z,Success,218
Bronze-to-Silver-WFP,manual_run,2026-10-07T18:56:24.085Z,Success,28026
Bronze-to-Silver-WFP,manual_run,2026-10-07T18:56:24.085Z,Failure,0
